In [1]:
import os, glob, json, time, numpy as np, pandas as pd, ee
from kaggle_secrets import UserSecretsClient

key = UserSecretsClient().get_secret("EE_KEY")
info = json.loads(key)
ee.Initialize(ee.ServiceAccountCredentials(info["client_email"], key_data=key), project="lulc-sar-project")
print("Earth Engine ready:", info["client_email"])

def f(name): return glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]
old = pd.read_csv(f("places_v2.csv"))
cit = pd.read_csv(f("cities_final.csv"), encoding="utf-8")
wet = pd.read_csv(f("wetlands_selected.csv"))
man = pd.read_csv(f("mangroves_selected.csv"))
print("places", len(old), "| cities", len(cit), "| wetlands", len(wet), "| mangroves", len(man))

Earth Engine ready: kaggle-ee@lulc-sar-project.iam.gserviceaccount.com
places 1000 | cities 41 | wetlands 23 | mangroves 7


In [2]:
import math

# 1) exclusion list
excl = (list(zip(old.lat, old.lon)) + list(zip(cit.lat, cit.lon))
        + list(zip(wet.lat, wet.lon)) + list(zip(man.lat, man.lon))
        + [(1.3521, 103.8198), (-36.8485, 174.7633)])
ex = np.radians(np.array(excl))
print("exclusion points:", len(excl))

def min_dist_km(la, lo):
    p1, l1 = np.radians(la), np.radians(lo)
    a = np.sin((ex[:, 0] - p1) / 2) ** 2 + np.cos(p1) * np.cos(ex[:, 0]) * np.sin((ex[:, 1] - l1) / 2) ** 2
    return float((12742 * np.arcsin(np.sqrt(a))).min())

def hav(a, b, c, d):
    p = math.radians
    x = math.sin(p(c - a) / 2) ** 2 + math.cos(p(a)) * math.cos(p(c)) * math.sin(p(d - b) / 2) ** 2
    return 12742 * math.asin(math.sqrt(x))

# 2) water share per 10x10 km box (WorldCover: 80 = permanent water; ocean is unmapped, so use "no data" as sea)
wc = ee.ImageCollection("ESA/WorldCover/v200").first()
land = wc.mask().rename("land")            # 1 where WorldCover has data (land), 0 over open sea

def measure_coast(chunk):
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la])) for la, lo in chunk])
    def g(feat):
        box = feat.geometry().buffer(5000).bounds()
        s = land.unmask(0).reduceRegion(ee.Reducer.mean(), box, scale=100, maxPixels=1e8).get("land")
        return feat.set("landshare", s)
    out = fc.map(g).getInfo()["features"]
    return [(o["geometry"]["coordinates"][1], o["geometry"]["coordinates"][0], o["properties"].get("landshare"))
            for o in out if o["properties"].get("landshare") is not None]

grid = [(float(la), float(lo)) for la in np.arange(-60, 78, 0.5) for lo in np.arange(-180, 180, 0.5)]
print("grid points:", len(grid))

# timing test on 500 points
t0 = time.time()
r = measure_coast(grid[30000:30500])
print("500 points took", round(time.time() - t0), "s; mixed (land share 0.25-0.75):",
      sum(0.25 <= x[2] <= 0.75 for x in r))

exclusion points: 1073
grid points: 198720
500 points took 2 s; mixed (land share 0.25-0.75): 4


In [3]:
rows, step, t0 = [], 500, time.time()
for i in range(0, len(grid), step):
    try:
        rows += [x for x in measure_coast(grid[i:i+step]) if 0.25 <= x[2] <= 0.75]
    except Exception as e:
        print("chunk", i, "failed:", str(e)[:80])
    if (i // step) % 40 == 0:
        print(i, "/", len(grid), "elapsed", round(time.time() - t0), "s, coastal rows", len(rows))
        pd.DataFrame(rows, columns=["lat", "lon", "landshare"]).to_csv("/kaggle/working/coast_grid.csv", index=False)

cg = pd.DataFrame(rows, columns=["lat", "lon", "landshare"])
cg.to_csv("/kaggle/working/coast_grid.csv", index=False)
print("coastal candidates:", len(cg))
print(cg.landshare.describe().round(2).to_string())

0 / 198720 elapsed 2 s, coastal rows 0
20000 / 198720 elapsed 84 s, coastal rows 18
40000 / 198720 elapsed 157 s, coastal rows 55
60000 / 198720 elapsed 238 s, coastal rows 139
80000 / 198720 elapsed 314 s, coastal rows 253
100000 / 198720 elapsed 388 s, coastal rows 392
120000 / 198720 elapsed 474 s, coastal rows 538
140000 / 198720 elapsed 551 s, coastal rows 615
160000 / 198720 elapsed 623 s, coastal rows 712
180000 / 198720 elapsed 698 s, coastal rows 856
coastal candidates: 1090
count    1090.00
mean        0.50
std         0.14
min         0.25
25%         0.38
50%         0.50
75%         0.62
max         0.75


In [4]:
CONT = lambda la, lo: ("Oceania" if (lo >= 110 and la < -10) else
    "North America" if (-170 <= lo <= -30 and la >= 12) else
    "South America" if (-92 <= lo <= -30 and la < 12) else
    "Europe" if (-25 <= lo <= 60 and 35 <= la <= 72) else
    "Africa" if (-20 <= lo <= 55 and -36 <= la < 35) else
    "Asia" if lo >= 25 else "Other")

c = cg.copy()
c["min_km"] = [min_dist_km(a, b) for a, b in zip(c.lat, c.lon)]
c = c[c.min_km >= 25].copy()
c["continent"] = [CONT(a, b) for a, b in zip(c.lat, c.lon)]
c = c[c.continent != "Other"]
c["band"] = pd.cut(c.lat.abs(), [0, 23, 45, 90], labels=["tropic", "mid", "high"])
print("usable coastal candidates:", len(c))
print(pd.crosstab(c.continent, c.band).to_string())

quota = {"Asia": 6, "Africa": 5, "North America": 5, "South America": 4, "Europe": 5, "Oceania": 5}
rng = np.random.default_rng(21)
chosen = []
def ok(r):
    return all(hav(r.lat, r.lon, k.lat, k.lon) >= 150 for k in chosen)

for cont, q in quota.items():
    p = c[c.continent == cont].copy()
    p["rnd"] = rng.random(len(p))
    p = p.sort_values("rnd")
    n = 0
    for b in ["tropic", "mid", "high"] * 6:      # rotate over latitude bands
        if n >= q: break
        for r in p[p.band == b].itertuples():
            if not any(hav(r.lat, r.lon, k.lat, k.lon) < 1 for k in chosen) and ok(r):
                chosen.append(r); n += 1; break

cs = pd.DataFrame([k._asdict() for k in chosen]).drop(columns="Index")
cs = cs[["lat", "lon", "continent", "band", "landshare", "min_km"]]
cs.insert(0, "id", range(1072, 1072 + len(cs)))
cs["group"] = "coastal"
cs.to_csv("/kaggle/working/coastal_selected.csv", index=False)
print(len(cs), "selected")
print(cs.continent.value_counts().to_string())
print(cs.round(3).to_string(index=False))
print("\n----- CSV TEXT (copy this if the session resets) -----")
print(cs.round(4).to_csv(index=False))

usable coastal candidates: 990
band           tropic  mid  high
continent                       
Africa             53   25     0
Asia              222   42   124
Europe              0   39    88
North America      48   28   188
Oceania            42   25     1
South America      24   19    15
30 selected
continent
Asia             6
Africa           5
North America    5
Europe           5
Oceania          5
South America    4
  id   lat    lon     continent   band  landshare   min_km   group
1072  -7.5  128.5          Asia tropic      0.302  836.113 coastal
1073  42.5  132.5          Asia    mid      0.468  241.021 coastal
1074  61.0  159.5          Asia   high      0.448  867.086 coastal
1075   2.5   97.0          Asia tropic      0.465  414.140 coastal
1076  29.5  130.0          Asia    mid      0.647  840.436 coastal
1077  59.0  165.0          Asia   high      0.670 1244.436 coastal
1078 -17.0   39.5        Africa tropic      0.376  216.334 coastal
1079  28.0   51.0        Africa  

In [6]:
import glob
cs = pd.read_csv(glob.glob("/kaggle/input/**/coastal_selected.csv", recursive=True)[0])
new = pd.concat([
    cs[["id","lat","lon","continent","group"]],
    wet[["id","lat","lon","continent","group"]],
    man[["id","lat","lon","continent","group"]],
], ignore_index=True)
print(len(new), new.group.value_counts().to_dict())

months = [(2025, m) for m in range(1, 13)] + [(2026, m) for m in range(1, 10)]
dw_all = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterDate("2025-01-01", "2026-11-01").select("label").mode())
names = ["water","trees","grass","flooded","crops","shrub","built","bare","snow"]

def s1_counts(la, lo):
    pt = ee.Geometry.Point([lo, la])
    base = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(pt)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
    out = []
    for y, m in months:
        a = f"{y}-{m:02d}-01"
        b = f"{y+1}-01-01" if m == 12 else f"{y}-{m+1:02d}-01"
        out.append(base.filterDate(a, b).size())
    return ee.List(out)

def dw_hist(la, lo):
    box = ee.Geometry.Point([lo, la]).buffer(5000).bounds()
    return dw_all.reduceRegion(ee.Reducer.frequencyHistogram(), box, scale=100, maxPixels=1e8).get("label")

# test on the first 3 tiles only
for r in new.head(3).itertuples():
    c = s1_counts(r.lat, r.lon).getInfo()
    h = dw_hist(r.lat, r.lon).getInfo()
    print(r.id, r.group, "S1 zero months:", sum(x == 0 for x in c), "mean:", round(sum(c)/len(c), 1), "| DW hist:", h)

60 {'coastal': 30, 'wetland': 23, 'mangrove': 7}
1072 coastal S1 zero months: 0 mean: 3.1 | DW hist: {'0': 10049.85098039218}
1073 coastal S1 zero months: 10 mean: 1.3 | DW hist: {'0': 13514.580392156879}
1074 coastal S1 zero months: 0 mean: 4.3 | DW hist: {'0': 20552.027450980357}


In [7]:
rows = []
t0 = time.time()
for k, r in enumerate(new.itertuples()):
    try:
        c = s1_counts(r.lat, r.lon).getInfo()
        h = dw_hist(r.lat, r.lon).getInfo() or {}
        tot = sum(h.values()) or 1
        sh = {names[int(i)]: v / tot for i, v in h.items()}
        rows.append(dict(id=r.id, group=r.group, lat=r.lat, lon=r.lon,
                         s1_zero=sum(x == 0 for x in c), s1_mean=round(sum(c)/len(c), 1),
                         **{n: round(sh.get(n, 0), 3) for n in names}))
    except Exception as e:
        print(r.id, "failed:", str(e)[:60])
    if k % 10 == 0:
        print(k, "/", len(new), "elapsed", round(time.time() - t0), "s")
chk = pd.DataFrame(rows)
chk.to_csv("/kaggle/working/new_tiles_check.csv", index=False)
print(chk.drop(columns=["lat", "lon"]).to_string(index=False))
print("\n----- CSV TEXT -----")
print(chk.to_csv(index=False))

0 / 60 elapsed 1 s
10 / 60 elapsed 40 s
20 / 60 elapsed 61 s
30 / 60 elapsed 81 s
40 / 60 elapsed 106 s
50 / 60 elapsed 126 s
  id    group  s1_zero  s1_mean  water  trees  grass  flooded  crops  shrub  built  bare  snow
1072  coastal        0      3.1  1.000  0.000  0.000    0.000  0.000  0.000  0.000 0.000 0.000
1073  coastal       10      1.3  1.000  0.000  0.000    0.000  0.000  0.000  0.000 0.000 0.000
1074  coastal        0      4.3  1.000  0.000  0.000    0.000  0.000  0.000  0.000 0.000 0.000
1075  coastal        0      9.0  1.000  0.000  0.000    0.000  0.000  0.000  0.000 0.000 0.000
1076  coastal        0      4.0  1.000  0.000  0.000    0.000  0.000  0.000  0.000 0.000 0.000
1077  coastal        0      4.4  1.000  0.000  0.000    0.000  0.000  0.000  0.000 0.000 0.000
1078  coastal        0      6.7  1.000  0.000  0.000    0.000  0.000  0.000  0.000 0.000 0.000
1079  coastal        0      8.7  1.000  0.000  0.000    0.000  0.000  0.000  0.000 0.000 0.000
1080  coastal      

In [8]:
dwm_ = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterDate("2025-01-01", "2026-11-01")
        .select("label").mode().eq(0).rename("w"))

def measure_w2(chunk):
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la])) for la, lo in chunk])
    def g(feat):
        box = feat.geometry().buffer(5000).bounds()
        s = dwm_.reduceRegion(ee.Reducer.mean(), box, scale=200, maxPixels=1e8).get("w")
        return feat.set("w", s)
    out = fc.map(g).getInfo()["features"]
    return [(o["geometry"]["coordinates"][1], o["geometry"]["coordinates"][0], o["properties"].get("w"))
            for o in out if o["properties"].get("w") is not None]

# test on the points that WorldCover already flagged as land-sea mixed (the 1090 coast candidates)
t0 = time.time()
pts_c = list(zip(cg.lat, cg.lon))[:250]
r = measure_w2(pts_c)
print("250 points took", round(time.time()-t0), "s; DW water share 0.25-0.75:", sum(0.25 <= x[2] <= 0.75 for x in r), "of", len(r))

250 points took 14 s; DW water share 0.25-0.75: 0 of 249


In [9]:
offs = [-0.1, -0.05, 0.0, 0.05, 0.1]
def best_mix(la, lo):
    pts_ = [(la + dy, lo + dx) for dy in offs for dx in offs]
    r = measure_w2(pts_)
    return min(r, key=lambda x: abs(x[2] - 0.5))   # (lat, lon, water share)

test = cg.sample(20, random_state=1)
t0 = time.time()
res = [best_mix(r.lat, r.lon) for r in test.itertuples()]
print("took", round(time.time() - t0), "s")
print("closest-to-50% water share per candidate:", [round(x[2], 2) for x in res])
print("mixed (0.25-0.75):", sum(0.25 <= x[2] <= 0.75 for x in res), "of", len(res))

took 53 s
closest-to-50% water share per candidate: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1]
mixed (0.25-0.75): 0 of 20


In [10]:
wc_land = ee.ImageCollection("ESA/WorldCover/v200").first().mask().rename("m")

def has_land(chunk):
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la])) for la, lo in chunk])
    def g(feat):
        s = wc_land.reduceRegion(ee.Reducer.mean(), feat.geometry().buffer(5000).bounds(), scale=200, maxPixels=1e8).get("m")
        return feat.set("m", s)
    out = fc.map(g).getInfo()["features"]
    return [(o["geometry"]["coordinates"][1], o["geometry"]["coordinates"][0]) for o in out
            if (o["properties"].get("m") or 0) > 0.5]

grid1 = [(float(la), float(lo)) for la in np.arange(-56, 78, 0.5) for lo in np.arange(-180, 180, 0.5)]
land_pts, t0 = [], time.time()
for i in range(0, len(grid1), 500):
    try: land_pts += has_land(grid1[i:i+500])
    except Exception as e: print("chunk", i, "failed")
    if (i // 500) % 50 == 0: print(i, "/", len(grid1), round(time.time()-t0), "s, land", len(land_pts))
pd.DataFrame(land_pts, columns=["lat","lon"]).to_csv("/kaggle/working/land_pts.csv", index=False)
print("land points:", len(land_pts))

0 / 192960 3 s, land 5
25000 / 192960 101 s, land 958
50000 / 192960 196 s, land 5734
75000 / 192960 284 s, land 12480
100000 / 192960 369 s, land 19458
125000 / 192960 455 s, land 29037
150000 / 192960 536 s, land 41665
175000 / 192960 703 s, land 58322
land points: 67950


In [11]:
lp = pd.read_csv("/kaggle/working/land_pts.csv")
s = set(zip(lp.lat.round(1), lp.lon.round(1)))
def near_sea(la, lo):
    return any((round(la+dy,1), round(lo+dx,1)) not in s
               for dy in (-0.5,0,0.5) for dx in (-0.5,0,0.5) if (dy,dx)!=(0,0))
edge = lp[[near_sea(a,b) for a,b in zip(lp.lat, lp.lon)]].reset_index(drop=True)
print("land points:", len(lp), "| next to sea:", len(edge))

# exclude anything within 25 km of existing places
edge["min_km"] = [min_dist_km(a, b) for a, b in zip(edge.lat, edge.lon)]
edge = edge[edge.min_km >= 25].reset_index(drop=True)
print("far enough from existing places:", len(edge))

rows, t0 = [], time.time()
pts_e = list(zip(edge.lat, edge.lon))
for i in range(0, len(pts_e), 250):
    try: rows += measure_w2(pts_e[i:i+250])
    except Exception as e: print("chunk", i, "failed")
    if (i // 250) % 10 == 0: print(i, "/", len(pts_e), round(time.time()-t0), "s")
em = pd.DataFrame(rows, columns=["lat","lon","water"])
em.to_csv("/kaggle/working/edge_water.csv", index=False)
print("measured:", len(em))
for lo_, hi_ in [(0.25,0.75),(0.35,0.65)]:
    print(f"water share {lo_}-{hi_}:", int(em.water.between(lo_, hi_).sum()))

land points: 67950 | next to sea: 9602
far enough from existing places: 9536
0 / 9536 14 s
2500 / 9536 272 s
5000 / 9536 602 s
7500 / 9536 816 s
measured: 9532
water share 0.25-0.75: 816
water share 0.35-0.65: 490


In [12]:
c = em[em.water.between(0.30, 0.70)].copy()
c["min_km"] = [min_dist_km(a, b) for a, b in zip(c.lat, c.lon)]
c = c[c.min_km >= 25]
c["continent"] = [CONT(a, b) for a, b in zip(c.lat, c.lon)]
c = c[c.continent != "Other"]
c["band"] = pd.cut(c.lat.abs(), [0, 23, 45, 90], labels=["tropic", "mid", "high"])
print("usable shoreline candidates:", len(c))
print(pd.crosstab(c.continent, c.band).to_string())

quota = {"Asia": 6, "Africa": 5, "North America": 5, "South America": 4, "Europe": 5, "Oceania": 5}
rng = np.random.default_rng(31)
chosen = []
def ok(r):
    return all(hav(r.lat, r.lon, k.lat, k.lon) >= 150 for k in chosen)

for cont, q in quota.items():
    p = c[c.continent == cont].copy()
    p["rnd"] = rng.random(len(p))
    p = p.sort_values("rnd")
    n = 0
    for b in ["tropic", "mid", "high"] * 8:
        if n >= q: break
        for r in p[p.band == b].itertuples():
            if ok(r) and not any(k.lat == r.lat and k.lon == r.lon for k in chosen):
                chosen.append(r); n += 1; break

cs2 = pd.DataFrame([k._asdict() for k in chosen]).drop(columns="Index")
cs2 = cs2[["lat", "lon", "continent", "band", "water", "min_km"]]
cs2.insert(0, "id", range(1072, 1072 + len(cs2)))
cs2["group"] = "coastal"
cs2.to_csv("/kaggle/working/coastal_selected_v2.csv", index=False)
print(len(cs2), "selected")
print(cs2.continent.value_counts().to_string())
print(cs2.round(3).to_string(index=False))
print("\n----- CSV TEXT -----")
print(cs2.round(4).to_csv(index=False))

usable shoreline candidates: 637
band           tropic  mid  high
continent                       
Africa             33   27     0
Asia              108   34    56
Europe              0   29    80
North America      20   24   130
Oceania            19   24     1
South America      26   10    13
30 selected
continent
Asia             6
Africa           5
North America    5
Europe           5
Oceania          5
South America    4
  id   lat    lon     continent   band  water  min_km   group
1072  20.0  110.5          Asia tropic  0.332 243.595 coastal
1073  45.0  147.5          Asia    mid  0.457 605.884 coastal
1074  68.5   74.5          Asia   high  0.325 162.989 coastal
1075  21.0  107.5          Asia tropic  0.591 243.319 coastal
1076  42.0  130.0          Asia    mid  0.466 427.011 coastal
1077  76.0   60.5          Asia   high  0.326 820.918 coastal
1078   5.0   -6.0        Africa tropic  0.315 476.642 coastal
1079 -25.0   47.0        Africa    mid  0.444  93.910 coastal
1080   4.

In [13]:
cs2 = pd.read_csv("/kaggle/working/coastal_selected_v2.csv")
rows = []
t0 = time.time()
for k, r in enumerate(cs2.itertuples()):
    try:
        c = s1_counts(r.lat, r.lon).getInfo()
        h = dw_hist(r.lat, r.lon).getInfo() or {}
        tot = sum(h.values()) or 1
        sh = {names[int(i)]: v / tot for i, v in h.items()}
        rows.append(dict(id=r.id, lat=r.lat, lon=r.lon,
                         s1_zero=sum(x == 0 for x in c), s1_mean=round(sum(c)/len(c), 1),
                         **{n: round(sh.get(n, 0), 3) for n in names}))
    except Exception as e:
        print(r.id, "failed:", str(e)[:60])
    if k % 10 == 0:
        print(k, "/", len(cs2), "elapsed", round(time.time() - t0), "s")
chk2 = pd.DataFrame(rows)
chk2.to_csv("/kaggle/working/coastal_check.csv", index=False)
print(chk2.drop(columns=["lat", "lon"]).to_string(index=False))
print("\n----- CSV TEXT -----")
print(chk2.to_csv(index=False))

0 / 30 elapsed 1 s
10 / 30 elapsed 25 s
20 / 30 elapsed 45 s
  id  s1_zero  s1_mean  water  trees  grass  flooded  crops  shrub  built  bare  snow
1072        0      6.9  0.329  0.217  0.014    0.021  0.082  0.007  0.317 0.013 0.000
1073        0      4.8  0.456  0.216  0.021    0.000  0.000  0.000  0.001 0.000 0.306
1074        3      3.5  0.320  0.028  0.184    0.000  0.000  0.015  0.000 0.007 0.445
1075        0      2.3  0.591  0.404  0.000    0.001  0.000  0.001  0.003 0.000 0.000
1076        0      4.6  0.462  0.291  0.002    0.003  0.198  0.025  0.014 0.005 0.000
1077       21      0.0  0.329  0.000  0.000    0.000  0.000  0.000  0.000 0.070 0.602
1078        0      3.0  0.315  0.612  0.003    0.015  0.000  0.041  0.013 0.001 0.000
1079        0      4.4  0.442  0.356  0.014    0.002  0.001  0.073  0.103 0.007 0.001
1080        0      5.4  0.456  0.403  0.000    0.138  0.000  0.000  0.000 0.000 0.002
1081        0     14.4  0.685  0.000  0.000    0.000  0.000  0.000  0.000 0.313

In [14]:
bad_ids = [1077, 1085]
keep = cs2[~cs2.id.isin(bad_ids)].copy()
pool = c[c.continent.isin(["Asia", "North America"]) & c.band.isin(["mid", "tropic"])].copy()
rng = np.random.default_rng(41)
pool["rnd"] = rng.random(len(pool))
new_rows = []
for need_cont in ["Asia", "North America"]:
    for r in pool[pool.continent == need_cont].sort_values("rnd").itertuples():
        pts_ = list(zip(keep.lat, keep.lon)) + [(x.lat, x.lon) for x in new_rows]
        if all(hav(r.lat, r.lon, a, b) >= 150 for a, b in pts_):
            cnt = s1_counts(r.lat, r.lon).getInfo()
            if sum(x == 0 for x in cnt) <= 4 and sum(cnt) / len(cnt) >= 2:
                new_rows.append(r); break
rep = pd.DataFrame([x._asdict() for x in new_rows]).drop(columns="Index")
rep = rep[["lat", "lon", "continent", "band", "water", "min_km"]]
rep.insert(0, "id", [1077, 1085][:len(rep)])
rep["group"] = "coastal"
final_coast = pd.concat([keep[["id","lat","lon","continent","band","water","min_km"]].assign(group="coastal"), rep]).sort_values("id")
final_coast.to_csv("/kaggle/working/coastal_final.csv", index=False)
print(len(final_coast), "coastal tiles")
print(rep.round(3).to_string(index=False))
print("\n----- CSV TEXT -----")
print(final_coast.round(4).to_csv(index=False))

AttributeError: 'list' object has no attribute 'continent'

In [15]:
cand_c = em[em.water.between(0.30, 0.70)].copy()
cand_c["min_km"] = [min_dist_km(a, b) for a, b in zip(cand_c.lat, cand_c.lon)]
cand_c = cand_c[cand_c.min_km >= 25].copy()
cand_c["continent"] = [CONT(a, b) for a, b in zip(cand_c.lat, cand_c.lon)]
cand_c = cand_c[cand_c.continent != "Other"]
cand_c["band"] = pd.cut(cand_c.lat.abs(), [0, 23, 45, 90], labels=["tropic", "mid", "high"])
print("candidates:", len(cand_c))

bad_ids = [1077, 1085]
keep = cs2[~cs2.id.isin(bad_ids)].copy()
pool = cand_c[cand_c.continent.isin(["Asia", "North America"]) & cand_c.band.isin(["mid", "tropic"])].copy()
rng = np.random.default_rng(41)
pool["rnd"] = rng.random(len(pool))
new_rows = []
for need_cont in ["Asia", "North America"]:
    for r in pool[pool.continent == need_cont].sort_values("rnd").itertuples():
        pts_ = list(zip(keep.lat, keep.lon)) + [(x.lat, x.lon) for x in new_rows]
        if all(hav(r.lat, r.lon, a, b) >= 150 for a, b in pts_):
            cnt = s1_counts(r.lat, r.lon).getInfo()
            if sum(x == 0 for x in cnt) <= 4 and sum(cnt) / len(cnt) >= 2:
                new_rows.append(r); break
rep = pd.DataFrame([x._asdict() for x in new_rows]).drop(columns="Index")
rep = rep[["lat", "lon", "continent", "band", "water", "min_km"]]
rep.insert(0, "id", [1077, 1085][:len(rep)])
rep["group"] = "coastal"
final_coast = pd.concat([keep[["id","lat","lon","continent","band","water","min_km"]].assign(group="coastal"), rep]).sort_values("id")
final_coast.to_csv("/kaggle/working/coastal_final.csv", index=False)
print(len(final_coast), "coastal tiles")
print(rep.round(3).to_string(index=False))
print("\n----- CSV TEXT -----")
print(final_coast.round(4).to_csv(index=False))

candidates: 637
30 coastal tiles
  id  lat   lon     continent   band  water  min_km   group
1077  9.5  80.5          Asia tropic  0.338 358.077 coastal
1085 20.0 -87.5 North America tropic  0.383  31.100 coastal

----- CSV TEXT -----
id,lat,lon,continent,band,water,min_km,group
1072,20.0,110.5,Asia,tropic,0.332,243.5955,coastal
1073,45.0,147.5,Asia,mid,0.457,605.884,coastal
1074,68.5,74.5,Asia,high,0.3245,162.9892,coastal
1075,21.0,107.5,Asia,tropic,0.5909,243.3192,coastal
1076,42.0,130.0,Asia,mid,0.466,427.0105,coastal
1077,9.5,80.5,Asia,tropic,0.3376,358.0765,coastal
1078,5.0,-6.0,Africa,tropic,0.3154,476.6418,coastal
1079,-25.0,47.0,Africa,mid,0.4441,93.9098,coastal
1080,4.5,7.0,Africa,tropic,0.4571,112.8663,coastal
1081,27.5,34.0,Africa,mid,0.6867,241.6536,coastal
1082,-23.0,43.5,Africa,tropic,0.4675,331.5205,coastal
1083,23.0,-80.5,North America,tropic,0.6975,216.3413,coastal
1084,24.5,-107.5,North America,mid,0.4398,35.2512,coastal
1085,20.0,-87.5,North America,tropic,0.383,31.0

In [16]:
lp = pd.read_csv("/kaggle/working/land_pts.csv")
pol = lp[(lp.lat >= 60) | (lp.lat <= -54)].copy()
pol["min_km"] = [min_dist_km(a, b) for a, b in zip(pol.lat, pol.lon)]
pol = pol[pol.min_km >= 25].reset_index(drop=True)
pol["continent"] = [CONT(a, b) for a, b in zip(pol.lat, pol.lon)]
print("polar land points:", len(pol))
print(pol.continent.value_counts().to_string())

# thin the pool: one random point per 2 degree cell, so the speed test covers the whole polar belt
pol["cell"] = list(zip((pol.lat // 2).astype(int), (pol.lon // 2).astype(int)))
pool_p = pol.sample(frac=1, random_state=5).drop_duplicates("cell").reset_index(drop=True)
print("one per 2-degree cell:", len(pool_p))
t0 = time.time()
r = [s1_counts(a, b).getInfo() for a, b in zip(pool_p.lat[:5], pool_p.lon[:5])]
print("5 S1 checks took", round(time.time() - t0), "s")

polar land points: 15920
continent
North America    6686
Asia             6513
Europe           2137
Other             509
South America      72
Oceania             3
one per 2-degree cell: 1201
5 S1 checks took 3 s


In [17]:
pool_p = pool_p.sample(frac=1, random_state=9).reset_index(drop=True)
pool_p["region"] = np.where(pool_p.lat < 0, "south", np.where(pool_p.continent == "Other", "greenland", pool_p.continent))
keep_rows, t0 = [], time.time()
for k, r in enumerate(pool_p.head(300).itertuples()):
    try:
        cnt = s1_counts(r.lat, r.lon).getInfo()
        z, m = sum(x == 0 for x in cnt), sum(cnt) / len(cnt)
        if z <= 4 and m >= 2:
            keep_rows.append((r.lat, r.lon, r.region, z, round(m, 1)))
    except Exception:
        pass
    if k % 50 == 0:
        print(k, "/ 300", round(time.time() - t0), "s, passing", len(keep_rows))
pp = pd.DataFrame(keep_rows, columns=["lat", "lon", "region", "s1_zero", "s1_mean"])
pp.to_csv("/kaggle/working/polar_s1_ok.csv", index=False)
print("passing S1 rule:", len(pp), "of 300")
print(pp.region.value_counts().to_string())
print("\n----- CSV TEXT -----")
print(pp.to_csv(index=False))

0 / 300 1 s, passing 1
50 / 300 30 s, passing 27
100 / 300 60 s, passing 49
150 / 300 81 s, passing 85
200 / 300 107 s, passing 114
250 / 300 125 s, passing 143
passing S1 rule: 168 of 300
region
Asia             66
North America    62
Europe           36
south             3
greenland         1

----- CSV TEXT -----
lat,lon,region,s1_zero,s1_mean
65.0,-153.5,North America,0,5.7
66.5,-98.5,North America,3,4.7
68.0,-145.0,North America,0,7.0
67.5,38.0,Europe,2,7.1
64.0,165.0,Asia,0,9.6
65.0,-131.5,North America,0,2.3
70.0,-132.0,North America,0,2.2
68.5,106.0,Asia,4,3.7
61.5,68.5,Asia,2,3.9
63.0,165.0,Asia,0,4.8
68.5,-160.0,North America,0,7.9
65.5,-150.5,North America,2,3.7
65.5,-21.0,Europe,0,14.7
75.0,142.0,Asia,1,3.4
62.0,31.5,Europe,0,20.7
70.5,57.5,Europe,4,3.0
70.0,-150.5,North America,0,5.4
73.0,89.5,Asia,3,3.7
65.5,92.5,Asia,3,4.0
63.0,-137.5,North America,0,4.9
66.0,178.5,Asia,0,7.0
60.0,57.5,Europe,3,2.1
64.5,-156.5,North America,0,6.0
70.5,69.0,Asia,2,3.9
65.0,53.5,Europe,3,3

In [18]:
pp = pd.read_csv("/kaggle/working/polar_s1_ok.csv")
pp["min_km"] = [min_dist_km(a, b) for a, b in zip(pp.lat, pp.lon)]
pp = pp[pp.min_km >= 25].copy()
rng = np.random.default_rng(51)
pp["rnd"] = rng.random(len(pp))
pp["sub"] = np.where(pp.lat < 0, "subantarctic",
            np.where(pp.region == "greenland", "svalbard",
            np.where(pp.lon.between(-170, -100) & (pp.lat > 0), "alaska_canada_west",
            np.where(pp.lon.between(-100, -50) & (pp.lat > 0), "canada_east",
            np.where(pp.lon.between(-30, -10) & (pp.lat > 0), "iceland",
            np.where(pp.lon.between(5, 45) & (pp.lat > 0), "scandinavia",
            np.where(pp.lon.between(45, 100), "siberia_west",
            np.where(pp.lon.between(100, 150), "siberia_central", "siberia_east"))))))))
quota = {"subantarctic": 3, "svalbard": 1, "alaska_canada_west": 3, "canada_east": 2, "iceland": 1,
         "scandinavia": 3, "siberia_west": 2, "siberia_central": 3, "siberia_east": 2}
chosen = []
for sub, q in quota.items():
    n = 0
    for r in pp[pp["sub"] == sub].sort_values("rnd").itertuples():
        if n >= q: break
        if all(hav(r.lat, r.lon, k.lat, k.lon) >= 150 for k in chosen):
            chosen.append(r); n += 1
ps = pd.DataFrame([k._asdict() for k in chosen]).drop(columns="Index")[["lat","lon","sub","s1_zero","s1_mean","min_km"]]

rows = []
for r in ps.itertuples():
    h = dw_hist(r.lat, r.lon).getInfo() or {}
    tot = sum(h.values()) or 1
    sh = {names[int(i)]: v / tot for i, v in h.items()}
    rows.append({n: round(sh.get(n, 0), 3) for n in names})
ps = pd.concat([ps.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
ps.insert(0, "id", range(1114, 1114 + len(ps)))
ps["group"] = "polar"
ps.to_csv("/kaggle/working/polar_selected.csv", index=False)
print(len(ps), "selected")
print(ps.drop(columns=["group"]).round(3).to_string(index=False))
print("\n----- CSV TEXT -----")
print(ps.round(4).to_csv(index=False))

20 selected
  id   lat    lon                sub  s1_zero  s1_mean   min_km  water  trees  grass  flooded  crops  shrub  built  bare  snow
1114 -54.5  -73.0       subantarctic        0     10.9  159.875  0.806  0.138  0.011    0.000  0.000  0.003  0.000 0.001 0.041
1115 -54.0  -70.0       subantarctic        2      9.4   47.777  0.047  0.844  0.001    0.000  0.000  0.068  0.000 0.000 0.039
1116 -54.5  -36.5       subantarctic        4      2.2 2097.823  0.212  0.001  0.000    0.000  0.000  0.000  0.000 0.002 0.786
1117  77.5   18.0           svalbard        3      2.1  868.054  0.320  0.000  0.000    0.000  0.000  0.000  0.000 0.000 0.680
1118  64.0 -144.5 alaska_canada_west        0     10.1  153.250  0.000  0.138  0.000    0.000  0.000  0.000  0.000 0.000 0.862
1119  68.0 -105.5 alaska_canada_west        4      3.5  110.473  0.166  0.001  0.021    0.000  0.000  0.525  0.000 0.000 0.287
1120  70.5 -149.5 alaska_canada_west        0      9.7  348.849  0.000  0.000  0.000    0.000  0.00

In [19]:
print([v in globals() for v in ["em", "lp", "min_dist_km", "CONT", "hav", "dwm_", "measure_w2", "s1_counts", "dw_hist"]])

[True, True, True, True, True, True, True, True, True]


In [20]:
lp = pd.read_csv("/kaggle/working/land_pts.csv")
s_ = set(zip(lp.lat.round(1), lp.lon.round(1)))
def near_sea_(la, lo):
    return any((round(la+dy,1), round(lo+dx,1)) not in s_
               for dy in (-1.0,-0.5,0,0.5,1.0) for dx in (-1.0,-0.5,0,0.5,1.0) if (dy,dx)!=(0,0))
inl = lp[[not near_sea_(a, b) for a, b in zip(lp.lat, lp.lon)]].copy()
print("inland land points (no sea within 1 deg):", len(inl))

inl["cell"] = list(zip(inl.lat.round(0), inl.lon.round(0)))
inl = inl.sample(frac=1, random_state=3).drop_duplicates("cell").reset_index(drop=True)
inl["min_km"] = [min_dist_km(a, b) for a, b in zip(inl.lat, inl.lon)]
inl = inl[inl.min_km >= 25].reset_index(drop=True)
print("one per 1-degree cell, far from existing places:", len(inl))

rows, t0 = [], time.time()
pts_i = list(zip(inl.lat, inl.lon))
for i in range(0, len(pts_i), 250):
    try: rows += measure_w2(pts_i[i:i+250])
    except Exception as e: print("chunk", i, "failed")
    if (i // 250) % 5 == 0: print(i, "/", len(pts_i), round(time.time()-t0), "s")
lk = pd.DataFrame(rows, columns=["lat","lon","water"])
lk.to_csv("/kaggle/working/lake_water.csv", index=False)
print("measured:", len(lk))
for lo_, hi_ in [(0.4,0.9),(0.5,0.9)]:
    print(f"water share {lo_}-{hi_}:", int(lk.water.between(lo_, hi_).sum()))
print("\n----- CSV TEXT (water share >= 0.4) -----")
print(lk[lk.water >= 0.4].round(3).to_csv(index=False))

inland land points (no sea within 1 deg): 52037
one per 1-degree cell, far from existing places: 13426
0 / 13426 16 s
1250 / 13426 122 s
2500 / 13426 209 s
3750 / 13426 299 s
5000 / 13426 411 s
6250 / 13426 505 s
7500 / 13426 619 s
8750 / 13426 716 s
10000 / 13426 808 s
11250 / 13426 896 s
12500 / 13426 997 s
measured: 13425
water share 0.4-0.9: 148
water share 0.5-0.9: 90

----- CSV TEXT (water share >= 0.4) -----
lat,lon,water
41.5,50.5,1.0
32.0,89.0,0.956
-49.0,-74.0,0.466
-0.5,33.5,1.0
56.5,-103.0,0.514
41.0,53.0,0.668
55.0,78.5,0.736
38.0,50.5,1.0
64.0,37.0,1.0
38.5,52.5,1.0
41.0,50.0,1.0
57.5,-6.0,0.852
-53.0,-71.5,1.0
43.0,49.0,1.0
0.0,32.5,0.987
-7.0,30.0,1.0
48.0,-99.0,0.464
41.5,-82.5,1.0
52.0,-76.0,0.526
55.5,109.5,1.0
53.5,49.0,0.464
48.0,-89.0,0.556
53.0,-101.0,0.498
38.0,53.5,1.0
42.5,-80.0,1.0
-1.5,-48.5,0.481
-8.5,26.5,0.592
7.5,-63.0,0.849
54.0,72.0,0.654
-13.0,-66.0,0.588
-2.0,-54.0,0.757
71.0,-53.0,1.0
50.5,-125.0,0.424
39.0,52.5,1.0
29.5,32.5,0.964
44.5,49.0,1.0
31.

In [21]:
L = lk[lk.water.between(0.40, 0.90)].copy()
L = L[~(L.lat.between(36, 47) & L.lon.between(46, 55))]
L["min_km"] = [min_dist_km(a, b) for a, b in zip(L.lat, L.lon)]
L = L[L.min_km >= 25]
L["continent"] = [CONT(a, b) for a, b in zip(L.lat, L.lon)]
L = L[L.continent != "Other"]
print(L.continent.value_counts().to_string())

quota = {"Africa": 2, "Asia": 2, "North America": 3, "South America": 1, "Europe": 2}
rng = np.random.default_rng(61)
L["rnd"] = rng.random(len(L))
chosen = []
for cont, q in quota.items():
    n = 0
    for r in L[L.continent == cont].sort_values("rnd").itertuples():
        if n >= q: break
        if not all(hav(r.lat, r.lon, k.lat, k.lon) >= 150 for k in chosen): continue
        cnt = s1_counts(r.lat, r.lon).getInfo()
        if sum(x == 0 for x in cnt) <= 4 and sum(cnt) / len(cnt) >= 2:
            chosen.append(r); n += 1

ls = pd.DataFrame([k._asdict() for k in chosen]).drop(columns="Index")[["lat", "lon", "continent", "water", "min_km"]]
rows = []
for r in ls.itertuples():
    h = dw_hist(r.lat, r.lon).getInfo() or {}
    tot = sum(h.values()) or 1
    sh = {names[int(i)]: v / tot for i, v in h.items()}
    rows.append({n: round(sh.get(n, 0), 3) for n in names})
ls = pd.concat([ls.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
ls.insert(0, "id", range(1134, 1134 + len(ls)))
ls["group"] = "lake"
ls.to_csv("/kaggle/working/lakes_selected.csv", index=False)
print(len(ls), "selected")
print(ls.drop(columns=["group"]).round(3).to_string(index=False))
print("\n----- CSV TEXT -----")
print(ls.round(4).to_csv(index=False))

continent
North America    56
Asia             39
Europe           23
South America    14
Africa            5
Oceania           5
10 selected
  id   lat    lon     continent  water  min_km  water  trees  grass  flooded  crops  shrub  built  bare  snow
1134 -10.0   34.5        Africa  0.500 148.736  0.498  0.421  0.000    0.000  0.000  0.075  0.005 0.000 0.000
1135  -9.0   29.0        Africa  0.552 169.130  0.553  0.255  0.006    0.002  0.000  0.176  0.006 0.002 0.000
1136  24.5   71.0          Asia  0.514 163.832  0.517  0.000  0.000    0.000  0.024  0.037  0.002 0.420 0.000
1137  25.0   91.0          Asia  0.752 380.732  0.732  0.038  0.002    0.024  0.174  0.003  0.025 0.002 0.000
1138  63.5 -101.0 North America  0.451 378.178  0.450  0.029  0.198    0.000  0.000  0.287  0.000 0.001 0.035
1139  58.5 -154.0 North America  0.416  79.775  0.417  0.302  0.016    0.007  0.000  0.007  0.000 0.010 0.241
1140  56.5 -133.0 North America  0.734 103.269  0.728  0.271  0.000    0.000  0.000  0.0